# Agentworld — exploration notebook

A working surface for poking at the smooth-striated continuum.

Read order:
1. The brief: `../README.md`
2. The variable: `../docs/concepts/smooth_striated.md`
3. The two attractors: `../docs/concepts/coasean_bargaining.md` and `../docs/concepts/fractal_folding.md`
4. This notebook

In [ ]:
%matplotlib inline
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import matplotlib.pyplot as plt
from engine.scenarios import SCENARIOS, get_scenario, list_scenarios
from engine.core.world import World
from engine.core.topology import TopologyConfig
from engine.core.population import PopulationConfig
from engine.core.world import WorldConfig

for n, d in list_scenarios():
    print(f'{n:30s} {d}')

## 1. The two attractors, side by side

Run Coasean Paradise and Baroque Cathedral and compare their EBI trajectories. This is the headline result of the artifact: the same agent layer, run at α=0.08 vs α=0.92, produces economies that diverge by ~3 orders of magnitude in nominal/real ratio.

In [ ]:
smooth = World.build(get_scenario('coasean_paradise'))
smooth.run(progress=False)
baroque = World.build(get_scenario('baroque_cathedral'))
baroque.run(progress=False)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, w, label in zip(axes, [smooth, baroque], ['Coasean Paradise (α=0.08)', 'Baroque Cathedral (α=0.92)']):
    h = w.metrics.history.to_dict()
    ax.plot(h['step'], h['real_welfare_cumulative'], label='real welfare', color='#2a8c4a')
    ax.plot(h['step'], h['nominal_gdp_cumulative'], label='nominal GDP', color='#c44d4d')
    ax.set_yscale('log')
    ax.set_title(label)
    ax.legend()
    ax.set_xlabel('step')
fig.tight_layout()

## 2. Sweep alpha continuously

Run a parameter sweep. For each α in [0.05, 0.95], do a short run, record the terminal EBI, fold depth, per-capita welfare. This produces the *response curve* of the model along the smooth-striated dimension.

In [ ]:
alphas = np.linspace(0.05, 0.95, 19)
results = []
for a in alphas:
    cfg = WorldConfig(
        population=PopulationConfig(seed=2024, n_human_prototypes=2000, n_agent_prototypes=20000),
        topology=TopologyConfig(alpha=float(a)),
        n_steps=30, pairs_per_step=80_000, seed=int(a * 1000),
    )
    w = World.build(cfg)
    w.run(progress=False)
    last = w.metrics.history.steps[-1]
    results.append({
        'alpha': a,
        'ebi': last.exo_baroque_index,
        'per_cap': last.real_per_capita_welfare,
        'fold_max': max(s.fold_max_depth for s in w.metrics.history.steps),
        'gov_oh': last.governance_overhead_fraction,
    })

import pandas as pd
df = pd.DataFrame(results)
df

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.6))
axes[0].plot(df['alpha'], df['ebi'], '-o', color='#6c3aa6')
axes[0].set_yscale('log'); axes[0].set_xlabel('α'); axes[0].set_ylabel('EBI'); axes[0].set_title('Exo-baroque index')
axes[0].axhline(1.0, color='#888', ls='--', lw=0.7)
axes[1].plot(df['alpha'], df['per_cap'], '-o', color='#2a8c4a')
axes[1].set_xlabel('α'); axes[1].set_ylabel('per-cap welfare'); axes[1].set_title('Welfare')
axes[2].plot(df['alpha'], df['fold_max'], '-o', color='#3a6ca6')
axes[2].set_xlabel('α'); axes[2].set_ylabel('max fold depth'); axes[2].set_title('Fold depth')
fig.tight_layout()

## 3. Capability vs α — the slop region

A 2D sweep. Holding α and capability mean both as variables, where does *welfare* peak vs where does *EBI* peak? The interesting region is the bottom-right: high α, low capability — what we call the **slop market**.


In [ ]:
alphas = np.linspace(0.1, 0.9, 7)
caps = np.linspace(0.4, 0.95, 6)
ebi_grid = np.zeros((len(caps), len(alphas)))
pc_grid = np.zeros((len(caps), len(alphas)))
for i, c in enumerate(caps):
    for j, a in enumerate(alphas):
        cfg = WorldConfig(
            population=PopulationConfig(
                agent_capability_mean=float(c), agent_capability_sd=0.10,
                human_capability_mean=float(max(0.2, c - 0.15)), human_capability_sd=0.10,
                seed=42, n_human_prototypes=1000, n_agent_prototypes=10000,
            ),
            topology=TopologyConfig(alpha=float(a)),
            n_steps=20, pairs_per_step=40_000, seed=42,
        )
        w = World.build(cfg); w.run(progress=False)
        ebi_grid[i, j] = w.metrics.history.steps[-1].exo_baroque_index
        pc_grid[i, j] = w.metrics.history.steps[-1].real_per_capita_welfare

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
im0 = axes[0].imshow(np.log10(ebi_grid), aspect='auto', origin='lower',
                     extent=[alphas[0], alphas[-1], caps[0], caps[-1]], cmap='viridis')
axes[0].set_title('log10(EBI) over (α, capability)')
axes[0].set_xlabel('α'); axes[0].set_ylabel('agent capability')
plt.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(pc_grid, aspect='auto', origin='lower',
                     extent=[alphas[0], alphas[-1], caps[0], caps[-1]], cmap='magma')
axes[1].set_title('per-capita welfare over (α, capability)')
axes[1].set_xlabel('α'); axes[1].set_ylabel('agent capability')
plt.colorbar(im1, ax=axes[1])
fig.tight_layout()

**Reading the heatmaps:**

- The EBI heatmap (left) shows the *Slop Market region* in the top-right and bottom-right: high α regimes amplify EBI regardless of capability, but low capability + high α produces the worst nominal/real ratio.
- The per-capita welfare heatmap (right) shows that *welfare peaks at high capability and low α* — the Coasean Paradise quadrant. Welfare is *not* maximized by either pure attractor; it is maximized by smoothing combined with capability uplift.

This is the key empirical finding the artifact surfaces: **the smooth attractor is welfare-maximizing only if capability is uniformly high.** With low capability, even smoothing produces only mediocre welfare; with high capability, the difference between smoothing and folding is the difference between welfare-and-GDP coupling vs. divergence.

## 4. The seam between stacks

Final exercise: how does cross-stack compatibility affect the system? Run hemispherical_schism with progressively lower compatibility values.

In [ ]:
compats = [0.85, 0.55, 0.30, 0.15, 0.05]
rows = []
for c in compats:
    cfg = WorldConfig(
        population=PopulationConfig(seed=314, stack_shares=(0.30, 0.30, 0.20, 0.20), n_human_prototypes=2000, n_agent_prototypes=20000),
        topology=TopologyConfig(alpha=0.55, cross_stack_compat=float(c), n_stacks=4),
        n_steps=30, pairs_per_step=80_000, seed=314,
    )
    w = World.build(cfg); w.run(progress=False)
    last = w.metrics.history.steps[-1]
    rows.append({'compat': c, 'ebi': last.exo_baroque_index, 'per_cap': last.real_per_capita_welfare, 'gov_oh': last.governance_overhead_fraction})

import pandas as pd
df_seam = pd.DataFrame(rows)
print(df_seam.to_string(index=False))

Lower cross-stack compatibility raises governance overhead and depresses welfare. EBI changes only slightly: cross-stack incompatibility is a *welfare* problem more than a *legibility* problem. Stack balkanization is a tax on real surplus, not on the fold-canopy.

## What to look at next

- The dashboard: `dashboard/index.html`. Open it directly in a browser.
- The figures: `outputs/figures/`. One per scenario plus the atlas and the alpha-sweep.
- The brief: `README.md`.
- The Cursor canvas: `review/agentworld.canvas.tsx`. The executive view.
